In [1]:
import os
import torch

os.chdir('C:\\Users\\pakhrins\\transptm_backup\\')

data = torch.load("transptm_final_acetylation_graph_data_made_by_subash.pt", weights_only=False)

C:\Users\pakhrins\AppData\Local\Subash_anaconda3\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  import torch_geometric.typing


In [7]:
import pandas as pd

df = pd.read_csv("NHAC_deduplicated.csv")

valid_data = [item for item in data[0] if item.emb.shape[0] == 25]

df_train = df[df["set"] == "train"]
train_identifier = set(df_train["unique_id"])

df_val = df[df["set"] == "val"]
val_identifier = set(df_val["unique_id"])

df_test = df[df["set"] == "test"]
test_identifier = set(df_test["unique_id"])

test_data = []
for i in range(len(valid_data)):
    if valid_data[i].unique_id in test_identifier:
        test_data.append(valid_data[i])     

test_embs = torch.stack([item.emb for item in test_data])
test_ys = torch.tensor([item.y for item in test_data])

num_ones = (test_ys == 1).sum().item()
num_zeros = (test_ys == 0).sum().item()

print("Information about Test Datasets")
print("Number of 1s:", num_ones)
print("Number of 0s:", num_zeros)
print()
val_data = []
for i in range(len(valid_data)):
    if valid_data[i].unique_id in val_identifier:
        val_data.append(valid_data[i])     

val_embs = torch.stack([item.emb for item in val_data])
val_ys = torch.tensor([item.y for item in val_data])

num_ones = (val_ys == 1).sum().item()
num_zeros = (val_ys == 0).sum().item()

print("\nInformation about Validation Datasets")
print("Number of 1s:", num_ones)
print("Number of 0s:", num_zeros)

train_data = []
for i in range(len(valid_data)):
    if valid_data[i].unique_id in train_identifier:
        train_data.append(valid_data[i])     

train_embs = torch.stack([item.emb for item in train_data])
train_ys = torch.tensor([item.y for item in train_data])

num_ones = (train_ys == 1).sum().item()
num_zeros = (train_ys == 0).sum().item()

print("\nInformation about Training Datasets")
print("Number of 1s:", num_ones)
print("Number of 0s:", num_zeros)

Information about Test Datasets
Number of 1s: 113
Number of 0s: 936


Information about Validation Datasets
Number of 1s: 87
Number of 0s: 376

Information about Training Datasets
Number of 1s: 586
Number of 0s: 3376


In [8]:
from torch.utils.data import Dataset, DataLoader

class PTMDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return self.y.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = PTMDataset(train_embs, train_ys)
val_ds   = PTMDataset(val_embs, val_ys)
test_ds  = PTMDataset(test_embs, test_ys)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=128)
test_loader  = DataLoader(test_ds, batch_size=128)

In [9]:
import torch.nn as nn
import numpy as np
import random
import gc
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import matthews_corrcoef, roc_auc_score, confusion_matrix

In [10]:
random.seed(7)
np.random.seed(7)
torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import matthews_corrcoef, roc_auc_score, confusion_matrix

# ============================================================
# Seeds
# ============================================================

random.seed(7)
np.random.seed(7)
torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ============================================================
# Model — SMALLER to prevent overfitting
# d_model: 512 -> 256
# num_layers: 3 -> 2
# dropout: 0.3 -> 0.5
# ============================================================

class LearnablePositionalEncoding(nn.Module):
    def __init__(self, seq_len, d_model):
        super().__init__()
        self.pos_embedding = nn.Parameter(torch.randn(1, seq_len, d_model))

    def forward(self, x):
        return x + self.pos_embedding


class AcetylationTransformer(nn.Module):
    def __init__(
        self,
        seq_len=25,
        input_dim=1024,
        d_model=256,          # was 512
        num_heads=8,
        num_layers=2,         # was 3
        dim_feedforward=512,  # was 1024
        dropout=0.5,          # was 0.3
    ):
        super().__init__()

        self.input_projection = nn.Linear(input_dim, d_model)
        self.input_norm = nn.LayerNorm(d_model)
        self.pos_encoding = LearnablePositionalEncoding(seq_len, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        # center + mean + max
        self.classifier = nn.Sequential(
            nn.Linear(d_model * 3, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        x = self.input_projection(x)
        x = self.input_norm(x)
        x = self.pos_encoding(x)
        x = self.transformer_encoder(x)

        center_index = x.shape[1] // 2
        center_feat = x[:, center_index, :]
        mean_feat   = x.mean(dim=1)
        max_feat    = x.max(dim=1).values

        x = torch.cat([center_feat, mean_feat, max_feat], dim=1)
        return self.classifier(x).squeeze(-1)


model = AcetylationTransformer().to(device)

# ============================================================
# Class counts
# ============================================================

pos_count = train_ys.sum().item()
neg_count = len(train_ys) - pos_count
print(f"Positives: {pos_count}  Negatives: {neg_count}  Ratio: {neg_count/pos_count:.2f}x")

# ============================================================
# Focal Loss — back to full pos_weight, gamma=2.0
# (v1 had the best AUC, we keep those loss settings)
# ============================================================

pos_weight = torch.tensor([neg_count / pos_count]).to(device)

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, pos_weight=None):
        super().__init__()
        self.gamma = gamma
        self.pos_weight = pos_weight

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pos_weight, reduction='none')
        pt = torch.sigmoid(logits)
        pt = torch.where(targets == 1, pt, 1 - pt)
        return ((1 - pt) ** self.gamma * bce).mean()

criterion = FocalLoss(gamma=2.0, pos_weight=pos_weight)

# ============================================================
# Weighted Sampler — full ratio (same as v1 which gave best SN)
# ============================================================

sample_weights = torch.where(
    train_ys == 1,
    torch.tensor(neg_count / pos_count),
    torch.tensor(1.0)
)
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
train_loader = DataLoader(train_ds, batch_size=128, sampler=sampler)

# ============================================================
# Optimizer — higher weight decay to fight overfitting
# ============================================================

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)  # was 1e-4
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

# ============================================================
# Training Functions
# ============================================================

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0
    for X, y in loader:
        X = X.to(device)
        y = y.float().to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate_probs(model, loader):
    model.eval()
    all_labels, all_probs = [], []
    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            y = y.float().to(device)
            probs = torch.sigmoid(model(X))
            all_labels.extend(y.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    return np.array(all_labels), np.array(all_probs)

# ============================================================
# Training Loop — Early Stopping on Val AUC
# ============================================================

best_auc = 0
patience = 10       # more patience since model trains slower now
counter  = 0
num_epochs = 80     # allow more epochs since model is smaller + regularized

for epoch in range(num_epochs):

    train_loss = train_one_epoch(model, train_loader)

    val_labels, val_probs = evaluate_probs(model, val_loader)
    val_preds = (val_probs > 0.5).astype(int)

    val_mcc = matthews_corrcoef(val_labels, val_preds)
    val_auc = roc_auc_score(val_labels, val_probs)

    print(f"Epoch {epoch+1}  |  Train Loss: {train_loss:.4f}  |  Val MCC: {val_mcc:.4f}  |  Val AUC: {val_auc:.4f}")

    scheduler.step()

    if val_auc > best_auc:
        best_auc = val_auc
        counter  = 0
        torch.save(model.state_dict(), "best_model_v3.pt")
        print(f"  >>> Best model saved (AUC: {best_auc:.4f})")
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered.")
            break

# ============================================================
# Load Best Model
# ============================================================

model.load_state_dict(torch.load("best_model_v3.pt"))
print(f"\nLoaded best model — Val AUC: {best_auc:.4f}")

# ============================================================
# Threshold Optimization on Validation Set
# ============================================================

val_labels, val_probs = evaluate_probs(model, val_loader)

best_thresh  = 0.5
best_val_mcc = -1

for t in np.linspace(0.1, 0.9, 200):
    preds = (val_probs > t).astype(int)
    mcc = matthews_corrcoef(val_labels, preds)
    if mcc > best_val_mcc:
        best_val_mcc = mcc
        best_thresh  = t

print(f"Best Threshold: {best_thresh:.4f}  (Val MCC: {best_val_mcc:.4f})")

# ============================================================
# Final Test Evaluation
# ============================================================

test_labels, test_probs = evaluate_probs(model, test_loader)
test_preds = (test_probs > 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(test_labels, test_preds).ravel()

acc = (tp + tn) / (tp + tn + fp + fn)
sn  = tp / (tp + fn)  if (tp + fn) != 0 else 0
sp  = tn / (tn + fp)  if (tn + fp) != 0 else 0
numerator   = (tp * tn) - (fp * fn)
denominator = np.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
mcc = numerator / denominator if denominator != 0 else 0
auc = roc_auc_score(test_labels, test_probs)

print("\n===== FINAL TEST RESULTS =====")
print(f"ACC: {acc:.4f}")
print(f"SN:  {sn:.4f}")
print(f"SP:  {sp:.4f}")
print(f"MCC: {mcc:.4f}")
print(f"AUC: {auc:.4f}")
print(f"TP: {tp}  FP: {fp}")
print(f"FN: {fn}  TN: {tn}")

Using device: cpu
Positives: 586.0  Negatives: 3376.0  Ratio: 5.76x
Epoch 1  |  Train Loss: 0.3940  |  Val MCC: 0.0000  |  Val AUC: 0.8040
  >>> Best model saved (AUC: 0.8040)
Epoch 2  |  Train Loss: 0.3135  |  Val MCC: 0.1870  |  Val AUC: 0.8037
Epoch 3  |  Train Loss: 0.2816  |  Val MCC: 0.2771  |  Val AUC: 0.8158
  >>> Best model saved (AUC: 0.8158)
Epoch 4  |  Train Loss: 0.2580  |  Val MCC: 0.2857  |  Val AUC: 0.8218
  >>> Best model saved (AUC: 0.8218)
Epoch 5  |  Train Loss: 0.2296  |  Val MCC: 0.3778  |  Val AUC: 0.8199
Epoch 6  |  Train Loss: 0.2091  |  Val MCC: 0.3473  |  Val AUC: 0.8192
Epoch 7  |  Train Loss: 0.1735  |  Val MCC: 0.3280  |  Val AUC: 0.8232
  >>> Best model saved (AUC: 0.8232)
Epoch 8  |  Train Loss: 0.1545  |  Val MCC: 0.3659  |  Val AUC: 0.8139
Epoch 9  |  Train Loss: 0.1336  |  Val MCC: 0.3756  |  Val AUC: 0.8170
Epoch 10  |  Train Loss: 0.1195  |  Val MCC: 0.3206  |  Val AUC: 0.8086
Epoch 11  |  Train Loss: 0.1121  |  Val MCC: 0.3637  |  Val AUC: 0.8132
E

# This is a decent result from transformers

# GNN

In [20]:
import torch
from torch import nn
import torch.nn.functional as F
from torch_scatter import scatter
from torch_geometric.nn import TransformerConv, GCNConv, GATConv


class GNNTrans(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers):
        super().__init__()
        self.num_layers = num_layers
        self.loss_func = nn.BCELoss()
        self.convs = torch.nn.ModuleList(
            [TransformerConv(in_channels=input_dim, out_channels=hidden_dim, heads=1)] +
            [TransformerConv(in_channels=hidden_dim, out_channels=hidden_dim, heads=1)
                for _ in range(num_layers-1)]
        )

        self.mlp = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(hidden_dim, hidden_dim ),  # Match this input dimension
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(hidden_dim, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def get_conv_result(self, x, edge_index1):
        for i in range(self.num_layers):
            x = self.convs[i](x=x, edge_index=edge_index1)
            x = F.relu(x, inplace=True)
        return x

    def forward(self, data):
        x, edge_index, batch = data.emb, data.edge_index1, data.batch



        conv_output = self.get_conv_result(x, edge_index)
        self.feature_responses_conv = conv_output[1].detach().cpu().numpy()

        idx = (data.ptr + int(len(data.seq[0]) / 2))[:-1]
        x = self.get_conv_result(x, edge_index)
        x = x[idx]

        linear_output = self.mlp[:-2](x)
        self.feature_responses_linear = linear_output.detach().cpu().numpy()

        x = self.mlp(x)

        return x

    def loss(self, pred, label):
        pred, label = pred.reshape(-1), label.reshape(-1)
        return self.loss_func(pred, label)

In [18]:
# import torch
# print(torch.__version__)
# print(torch.version.cuda)

In [19]:
# !pip install torch-scatter -f https://data.pyg.org/whl/torch-2.2.0+cu121.html
# !pip install torch-sparse -f https://data.pyg.org/whl/torch-2.2.0+cu121.html
# !pip install torch-cluster -f https://data.pyg.org/whl/torch-2.2.0+cu121.html
# !pip install torch-geometric

In [21]:
import time
import torch
import torch.optim as optim
import copy
import numpy as np
from torch import nn
from types import SimpleNamespace
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.metrics import matthews_corrcoef
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score

##things I am adding
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score


class TrainProcessor:
    def __init__(self, model, loaders, args, save_train_outputs=False):
        self.model = model
        self.train_loader, self.val_loader, self.test_loader = loaders
        print('len train_loader.dataset:', len(self.train_loader.dataset))
        print('len val_loader.dataset:', len(self.val_loader.dataset))
        print('len test_loader.dataset:', len(self.test_loader.dataset))
        self.args = args
        self.optimizer, self.scheduler = self.build_optimizer()
        self.save_train_outputs = save_train_outputs

    def build_optimizer(self):
        args = self.args
        # return an iterator
        filter_fn = filter(lambda p: p.requires_grad, self.model.parameters())  # params is a generator (kind of iterator)

        # optimizer
        weight_decay = args.weight_decay
        if args.opt == 'adam':
            optimizer = optim.Adam(filter_fn, lr=args.lr, weight_decay=weight_decay)
        elif args.opt == 'sgd':
            optimizer = optim.SGD(filter_fn, lr=args.lr, momentum=0.95, weight_decay=weight_decay)
        elif args.opt == 'rmsprop':
            optimizer = optim.RMSprop(filter_fn, lr=args.lr, weight_decay=weight_decay)
        elif args.opt == 'adagrad':
            optimizer = optim.Adagrad(filter_fn, lr=args.lr, weight_decay=weight_decay)

        # scheduler
        if args.opt_scheduler == 'none':
            return None, optimizer
        elif args.opt_scheduler == 'step':
            scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=args.opt_decay_step, gamma=args.opt_decay_rate)
        elif args.opt_scheduler == 'reduceOnPlateau':
            scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                             patience=args.lr_decay_patience,
                                                             factor=args.lr_decay_factor)
        else:
            raise Exception('Unknown optimizer type')

        return optimizer, scheduler



    @torch.no_grad()
    def test(self, model, dataloader):

        model.eval()


        pred_ls = []
        y_ls = []
        for batch in dataloader:
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            batch = batch.to(self.args.device)
            pred_ls.append(model(batch))
            y_ls.append(batch.y)

        pred_transptm = torch.cat(pred_ls, dim=0).reshape(-1)  # 预测出来的y=1的概率
        y = torch.cat(y_ls, dim=0).reshape(-1)  # 真实的标签

        # get metrics
        metrics = {}
        metrics['loss'] = model.loss(pred_transptm, y).item()
        metrics['acc'] = (pred_transptm.round() == y).sum() / len(pred_transptm)




        y = y.detach().cpu().numpy()
        pred_transptm = pred_transptm.detach().cpu().numpy()

        metrics['auroc'] = roc_auc_score(y, pred_transptm)
        metrics['auprc'] = average_precision_score(y, pred_transptm)
        metrics['MCC'] = matthews_corrcoef(y, pred_transptm.round())
        metrics['f1'] = f1_score(y, pred_transptm.round())
        metrics['precision'] = precision_score(y, pred_transptm.round(), zero_division=1)
        metrics['recall'] = recall_score(y, pred_transptm.round())
        metrics['my_accuracy'] = accuracy_score(y, pred_transptm.round())
        metrics['confusion_matrix'] = confusion_matrix(y, pred_transptm.round())

        #metrics['Specificity'] =


        return SimpleNamespace(**metrics)

    def train(self):

        best_val_loss = float('inf')
        best_model = None
        es = 0

        for epoch in range(self.args.epochs):

            epoch_lr = self.optimizer.param_groups[0]['lr']
            train_epoch_loss = 0.0
            self.model.train()

            for batch_idx, batch in enumerate(self.train_loader):
                batch = batch.to(self.args.device)
                self.optimizer.zero_grad()

                pred_transptm = self.model(batch)
                label = batch.y

                loss = self.model.loss(pred_transptm, label)
                loss.backward()

                # clip gradients
                # nn.utils.clip_grad_value_(self.model.parameters(), clip_value=1.0)

                self.optimizer.step()
                train_epoch_loss += loss.item()


            train_epoch_loss /= len(self.train_loader)

            # ---validation---
            val_metrics = self.test(self.model, self.val_loader)
            val_epoch_loss, val_epoch_roc = val_metrics.loss, val_metrics.auroc

            self.model.train()
            if self.args.opt_scheduler is None:
                pass
            elif self.args.opt_scheduler == 'reduceOnPlateau':
                self.scheduler.step(val_epoch_loss)
            elif self.args.opt_scheduler == 'step':
                self.scheduler.step()

            # print training process
            log = 'Epoch: {:03d}/{:03d}; ' \
                  'AVG Training Loss (MSE):{:.5f}; ' \
                  'AVG Val Loss (MSE):{:.5f};' \
                  'AVG Val AUROC:{:.5f};' \
                  'lr:{:8f}'
            print(time.strftime('%H:%M:%S'),
                  log.format(
                      epoch + 1,
                      self.args.epochs,
                      train_epoch_loss,
                      val_epoch_loss,
                      val_epoch_roc,
                      epoch_lr
                  ))
            if epoch_lr != self.optimizer.param_groups[0]['lr']:
                print('lr has been updated from {:.8f} to {:.8f}'.format(epoch_lr,
                                                                         self.optimizer.param_groups[0]['lr']))

            # determine whether stop early by val_epoch_loss
            if val_epoch_loss < best_val_loss:
                best_val_loss = val_epoch_loss
                best_model = copy.deepcopy(self.model)
                es = 0
            else:
                es += 1
                print("Counter {} of patience {}".format(es, self.args.es_patience))
                if es >= self.args.es_patience:
                    print("Early stopping with best_val_loss {:.8f}".format(best_val_loss))
                    break

        test_metrics = self.test(best_model, self.test_loader)

        return best_model, test_metrics

In [ ]:
import torch
from types import SimpleNamespace
import numpy as np
import random
from torch_geometric.loader import DataLoader
import os
import pandas as pd


seed_values = [random.randint(1, 1000000) for _ in range(300)]

for i in range(len(seed_values)):
    print("\n\n\n")
    seed = seed_values[i]
    print("New run! Seed: ", seed)
    random.seed(seed)
    np.random.seed(seed)
    ## I added all this to make it reproducible
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    g = torch.Generator()
    g.manual_seed(seed)

    if __name__ == '__main__':
        # args
        args = {
              'epochs': 500,
            'batch_size': 64,
            'device': torch.device("cuda" if torch.cuda.is_available() else "cpu"),
            'opt': 'adam',
            'opt_scheduler': 'step',
            'opt_decay_step': 20,
            'opt_decay_rate': 0.7,
            'weight_decay': 1e-4,
            'lr': 3e-5,
            'es_patience': 20,
            'save': True
        }
        args = SimpleNamespace(**args)
        print(args)

        train_ls = data[0][:3975]
        val_ls = data[0][3975:4439]
        test_ls = data[0][4439:]

        train = [item for item in train_ls if item.emb.shape[0] == 25]
        val = [item for item in val_ls if item.emb.shape[0] == 25]
        test = [item for item in test_ls if item.emb.shape[0] == 25]

        train_data_loader = DataLoader(train, batch_size=args.batch_size, shuffle=True) ## added shuffler/generator part
        val_data_loader = DataLoader(val, batch_size=args.batch_size, shuffle=False)
        test_data_loader = DataLoader(test, batch_size=args.batch_size, shuffle=False)

        for i in range(1):
            model = GNNTrans(input_dim=1024, hidden_dim=64, num_layers=2)  # hidden_dim:[64, 128, 256, 512], num_layers:[2,3]
            model.to(args.device)
            print(model)

            train_val = TrainProcessor(
                    model=model,
                    loaders=[train_data_loader, val_data_loader, test_data_loader],
                    args=args
                                    )
            best_model, test_metrics = train_val.train()
            print('test loss: {:5f}; confusion matrix:\n{}\n; test MCC: {:4f}; my acc: {:4f}; test acc: {:4f}; test recall: {:4f}; test precision {:4f}; test F1 Score {:4f}; test auroc: {:4f}; test auprc: {:.4f}'.format(
                test_metrics.loss, test_metrics.confusion_matrix, test_metrics.MCC, test_metrics.my_accuracy, test_metrics.acc, test_metrics.recall, test_metrics.precision, test_metrics.f1, test_metrics.auroc, test_metrics.auprc))






New run! Seed:  123132
namespace(epochs=500, batch_size=64, device=device(type='cpu'), opt='adam', opt_scheduler='step', opt_decay_step=20, opt_decay_rate=0.7, weight_decay=0.0001, lr=3e-05, es_patience=20, save=True)
GNNTrans(
  (loss_func): BCELoss()
  (convs): ModuleList(
    (0): TransformerConv(1024, 64, heads=1)
    (1): TransformerConv(64, 64, heads=1)
  )
  (mlp): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=64, out_features=64, bias=True)
    (2): ReLU(inplace=True)
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=64, out_features=64, bias=True)
    (5): ReLU(inplace=True)
    (6): Dropout(p=0.3, inplace=False)
    (7): Linear(in_features=64, out_features=1, bias=True)
    (8): Sigmoid()
  )
)
len train_loader.dataset: 3962
len val_loader.dataset: 463
len test_loader.dataset: 1049
13:39:37 Epoch: 001/500; AVG Training Loss (MSE):0.64665; AVG Val Loss (MSE):0.64834;AVG Val AUROC:0.43811;lr:0.000030
13:39:55 Epoch: 002/500;